# Rigol MHO5104 Oscilloscope — USB Control Notebook

**Instrument:** Rigol MHO5104 (DHO5000/MHO5000 series, Centaurus platform)  
**Firmware:** 00.01.01  
**Interface:** USB direct (USB-TMC) — no GPIB adapter required  
**Command set:** SCPI — all commands verified empirically on this unit

---

> ## ✅ Verified Command Summary
> | Subsystem | Status | Notes |
> |---|---|---|
> | Identity / System | ✅ | `*IDN?`, `*OPT?`, `*CLS`, `*RST` |
> | Timebase | ✅ | `MAIN:SCALe`, `MAIN:OFFSet`, `MODE`, delay |
> | Channels 1–4 | ✅ | Scale, offset, coupling, probe, BWLimit, invert, unit, impedance |
> | Acquire | ✅ | TYPE, AVERages, MDEPth (1k/10k/100k/1M), SRATe read-only |
> | Trigger | ✅ | Edge, pulse; SWEep, HOLDoff, COUPling, FORCe |
> | Run/Stop/Single | ✅ | `:RUN`, `:STOP`, `:SINGle`, `:AUToscale`, `:CLEar` |
> | Measurements | ✅ | 17 confirmed mnemonics — see MEAS_ITEMS below |
> | Waveform capture | ✅ | BYTE format; YREFerence=128; preamble for scaling |
> | Screenshot | ✅ | `:DISPlay:DATA?` → 3 MB BMP; converted to PNG via Pillow |
> | Math | ✅ | `:MATH1:` prefix (bare `:MATH:` does NOT work) |
> | Cursor | ⚠️ | MODE works; use `:CURSor:ITEM?` for JSON state |
> | AWG | ⚠️ | FUNCtion, FREQuency confirmed; amplitude/output still unverified |
> | Decode | ❌ | Not exposed on this firmware |
>
> **Does NOT work on this firmware:**
> WORD waveform format · RAW waveform mode · `:CHANnel:LABel?`  
> `:SYSTem:OUTPut:TYPE?` · `:ACQuire:RESolution?`  
> Memory depths 500000, 5000000 (valid: 1000 / 10000 / 100000 / 1000000)
>
> **Screenshot note:** The scope sends a 3,072,054-byte 1280×800 24-bit BMP with
> an IEEE 488.2 block header. `read_raw()` stops early at the USB-TMC EOM bit
> regardless of declared size. `query_binary_values()` is used instead, which
> loops on the declared byte count. The raw BMP is then converted to PNG via Pillow.

---
**How to use:**
1. Edit the **Configuration Block** (Cell 2).
2. Run imports/connection (Cell 3) and function definitions (Cell 4).
3. Use the example cells or call functions directly.

## ⚙️ CONFIGURATION BLOCK — Edit All Parameters Here

In [ ]:
# ============================================================
#  USB CONNECTION  (USB-TMC direct — no GPIB adapter needed)
#  Update the serial number if connecting a different unit.
# ============================================================
VISA_ADDR  = "USB0::0x1AB1::0x0450::MHO5C283M0017::INSTR"
TIMEOUT_MS = 5000

# ============================================================
#  TIMEBASE
# ============================================================
TB_SCALE_S    = 1e-6     # Time per division (seconds)  e.g. 1e-6 = 1 µs/div
TB_OFFSET_S   = 0.0      # Horizontal offset (seconds)
TB_MODE       = 'MAIN'   # 'MAIN', 'XY', 'ROLL'

# ============================================================
#  CHANNEL DEFAULTS
# ============================================================
CH_SCALE_V    = 1.0      # Volts per division
CH_OFFSET_V   = 0.0      # Vertical offset (volts)
CH_COUPLING   = 'DC'     # 'DC', 'AC', 'GND'
CH_BWLIMIT    = 'OFF'    # 'OFF', '20M'
CH_PROBE      = 10       # Probe attenuation ratio  (1, 10, 100 ...)
CH_IMPEDANCE  = 'OMEG'   # 'OMEG' = 1 MΩ,  'FIFT' = 50 Ω
CH_UNIT       = 'VOLT'   # 'VOLT', 'AMP', 'WATT', 'UNKN'

# ============================================================
#  ACQUIRE
# ============================================================
ACQ_TYPE      = 'NORM'   # 'NORM', 'AVER', 'PEAK', 'HRES'
ACQ_AVERAGES  = 4        # Number of averages (used when ACQ_TYPE='AVER')
ACQ_MDEPTH    = 10000    # Memory depth: 1000 / 10000 / 100000 / 1000000

# ============================================================
#  TRIGGER
# ============================================================
TRIG_MODE     = 'EDGE'   # 'EDGE', 'PULS', 'SLOPE', 'VID'
TRIG_SWEEP    = 'AUTO'   # 'AUTO', 'NORM', 'SING'
TRIG_SRC      = 'CHAN1'  # 'CHAN1'..'CHAN4', 'EXT', 'AC'
TRIG_SLOPE    = 'POS'    # 'POS', 'NEG', 'RFAL'
TRIG_LEVEL_V  = 0.0      # Trigger level (volts)
TRIG_COUPLING = 'DC'     # 'DC', 'AC', 'LFR', 'HFR'
TRIG_HOLDOFF  = 8e-9     # Holdoff (seconds)

# ============================================================
#  WAVEFORM CAPTURE
# ============================================================
WAV_CHANNEL   = 'CHANnel1'  # 'CHANnel1'..'CHANnel4', 'MATH1'
WAV_POINTS    = 10000        # Points to capture (≤ ACQ_MDEPTH)

# ============================================================
#  MEASUREMENTS
#  Confirmed working mnemonics on this firmware.
#  9.9E+37 is the scope's "no valid signal" sentinel.
# ============================================================
MEAS_ITEMS = {
    'vmax':      'VMAX',      # Maximum voltage
    'vmin':      'VMIN',      # Minimum voltage
    'vamp':      'VAMP',      # Amplitude (Vtop - Vbase)
    'vtop':      'VTOP',      # Top voltage
    'vbase':     'VBASE',     # Base voltage
    'vavg':      'VAVG',      # Average voltage
    'vrms':      'VRMS',      # RMS voltage
    'period':    'PERiod',    # Period (seconds)
    'frequency': 'FREQuency', # Frequency (Hz)
    'rise_time': 'RTIMe',     # Rise time  (NOT RISetime)
    'fall_time': 'FTIMe',     # Fall time  (NOT FALLtime)
    'pos_width': 'PWIDth',    # Positive pulse width
    'neg_width': 'NWIDth',    # Negative pulse width
    'pos_duty':  'PDUT',      # Positive duty cycle  (NOT PDUTycycle)
    'neg_duty':  'NDUT',      # Negative duty cycle  (NOT NDUTycycle)
    'overshoot': 'OVERshoot', # Overshoot
    'preshoot':  'PREShoot',  # Preshoot
    'neg_edges': 'NEDG',      # Negative edge count
}
MEAS_INVALID = 9.9e37

# ============================================================
#  AWG (built-in function generator)
# ============================================================
AWG_FUNCTION  = 'SIN'    # 'SIN', 'SQU', 'RAMP', 'PULS', 'NOIS', 'DC'
AWG_FREQ_HZ   = 1000.0   # Frequency (Hz)

# ============================================================
#  FILE OUTPUT
# ============================================================
OUTPUT_DIR    = './data_scope'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("MHO5104 configuration loaded.")

## 📦 Imports & Instrument Connection

In [ ]:
import pyvisa
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import time
import datetime
import os
import io
from PIL import Image

rm = pyvisa.ResourceManager()
print("VISA resources:", rm.list_resources())

scope = rm.open_resource(VISA_ADDR)
scope.timeout          = TIMEOUT_MS
scope.read_termination  = '\n'
scope.write_termination = '\n'
scope.write('*CLS')
time.sleep(0.1)

idn = scope.query('*IDN?').strip()
print(f"Connected : {idn}")
opts = scope.query('*OPT?').strip()
print(f"Options   : {opts or '(none)'}")

# ── Internal helpers ──────────────────────────────────────────────────────────

def _q(cmd, timeout_ms=TIMEOUT_MS):
    """Query with timeout guard; returns None on failure."""
    old = scope.timeout
    scope.timeout = timeout_ms
    try:
        return scope.query(cmd).strip()
    except pyvisa.errors.VisaIOError:
        return None
    finally:
        scope.timeout = old

def _qf(cmd, timeout_ms=TIMEOUT_MS):
    """Query and parse to float; returns None on failure."""
    r = _q(cmd, timeout_ms)
    try:
        return float(r) if r is not None else None
    except ValueError:
        return None

def check_error():
    """Drain and print the instrument error queue. Returns list of strings."""
    errors = []
    for _ in range(20):
        try:
            scope.timeout = 2000
            r = scope.query(':SYSTem:ERRor?').strip()
            errors.append(r)
            if r.split(',')[0].strip() in ('0', '+0'):
                break
        except Exception:
            break
    scope.timeout = TIMEOUT_MS
    for e in errors:
        if e.split(',')[0].strip() not in ('0', '+0'):
            print(f"  ERR: {e}")
    return errors

print("\nReady.")

## 🔧 Function Definitions

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  SYSTEM / UTILITY
# ══════════════════════════════════════════════════════════════════════════════

def scope_reset():
    """Reset to factory defaults and clear status. Waits 3 s."""
    scope.write('*RST')
    scope.write('*CLS')
    time.sleep(3)
    print("MHO5104 reset complete.")

def scope_idn():
    """Return the IDN string."""
    r = _q('*IDN?')
    print(f"IDN: {r}")
    return r

def scope_status():
    """
    Read the status byte, event status register, and trigger status.

    Returns
    -------
    dict with keys: stb, esr, trigger_status
    """
    s = {
        'stb':            _q('*STB?'),
        'esr':            _q('*ESR?'),
        'trigger_status': _q(':TRIGger:STATus?'),
    }
    print(f"  STB={s['stb']}  ESR={s['esr']}  "
          f"TRIGger:STATus={s['trigger_status']}")
    return s


# ══════════════════════════════════════════════════════════════════════════════
#  RUN / STOP / SINGLE
# ══════════════════════════════════════════════════════════════════════════════

def scope_run():
    """Start continuous acquisition."""
    scope.write(':RUN')
    print("Scope: RUN")

def scope_stop():
    """Stop acquisition and freeze the display."""
    scope.write(':STOP')
    print("Scope: STOP")

def scope_single():
    """Arm for a single acquisition."""
    scope.write(':SINGle')
    print("Scope: SINGle")

def scope_autoscale():
    """Run auto-scale. Waits 3 s for the scope to settle."""
    scope.write(':AUToscale')
    time.sleep(3)
    print("Scope: AUToscale complete.")

def scope_clear():
    """Clear waveform display."""
    scope.write(':CLEar')
    print("Scope: CLEar")

def scope_force_trigger():
    """Force a trigger immediately."""
    scope.write(':TRIGger:FORCe')
    print("Scope: FORCe trigger")

def scope_wait_trigger(timeout_s=10):
    """
    Poll until trigger status reaches TD (triggered/done) or timeout.

    Parameters
    ----------
    timeout_s : Maximum seconds to wait

    Returns
    -------
    status : str or None — final trigger status string
    """
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        status = _q(':TRIGger:STATus?')
        if status in ('TD', 'STOP'):
            print(f"Trigger: {status}")
            return status
        time.sleep(0.1)
    print("WARNING: trigger wait timed out")
    return None


# ══════════════════════════════════════════════════════════════════════════════
#  TIMEBASE
# ══════════════════════════════════════════════════════════════════════════════

def set_timebase(scale_s=TB_SCALE_S, offset_s=TB_OFFSET_S, mode=TB_MODE):
    """
    Configure the horizontal timebase.

    Parameters
    ----------
    scale_s  : Time per division in seconds  (e.g. 1e-6 = 1 µs/div)
    offset_s : Horizontal offset in seconds
    mode     : 'MAIN', 'XY', 'ROLL'
    """
    scope.write(f':TIMebase:MODE {mode}')
    scope.write(f':TIMebase:MAIN:SCALe {scale_s:.6e}')
    scope.write(f':TIMebase:MAIN:OFFSet {offset_s:.6e}')
    print(f"Timebase: {scale_s*1e6:.3f} µs/div, "
          f"offset={offset_s*1e6:.3f} µs, mode={mode}")

def get_timebase():
    """
    Query current timebase settings.

    Returns
    -------
    dict with keys: scale_s, offset_s, mode
    """
    s = {
        'scale_s':  _qf(':TIMebase:MAIN:SCALe?'),
        'offset_s': _qf(':TIMebase:MAIN:OFFSet?'),
        'mode':     _q(':TIMebase:MODE?'),
    }
    print(f"Timebase: {s['scale_s']*1e6:.3f} µs/div, "
          f"offset={s['offset_s']*1e6:.3f} µs, mode={s['mode']}")
    return s


# ══════════════════════════════════════════════════════════════════════════════
#  CHANNEL
# ══════════════════════════════════════════════════════════════════════════════

def channel_on(n):
    """Enable display of channel n (1–4)."""
    scope.write(f':CHANnel{n}:DISPlay ON')
    print(f"CH{n}: ON")

def channel_off(n):
    """Disable display of channel n (1–4)."""
    scope.write(f':CHANnel{n}:DISPlay OFF')
    print(f"CH{n}: OFF")

def set_channel(
    n,
    scale_v=CH_SCALE_V,
    offset_v=CH_OFFSET_V,
    coupling=CH_COUPLING,
    bwlimit=CH_BWLIMIT,
    probe=CH_PROBE,
    impedance=CH_IMPEDANCE,
    unit=CH_UNIT,
    invert=False,
    display=True
):
    """
    Configure a single channel.

    Parameters
    ----------
    n         : Channel number 1–4
    scale_v   : Volts per division
    offset_v  : Vertical offset (volts)
    coupling  : 'DC', 'AC', 'GND'
    bwlimit   : 'OFF', '20M'
    probe     : Probe ratio (1, 10, 100, 1000)
    impedance : 'OMEG' (1 MΩ) or 'FIFT' (50 Ω)
    unit      : 'VOLT', 'AMP', 'WATT', 'UNKN'
    invert    : True/False
    display   : True/False
    """
    c = f':CHANnel{n}'
    scope.write(f'{c}:DISPlay {"ON" if display else "OFF"}')
    scope.write(f'{c}:SCALe {scale_v:.6e}')
    scope.write(f'{c}:OFFSet {offset_v:.6e}')
    scope.write(f'{c}:COUPling {coupling}')
    scope.write(f'{c}:BWLimit {bwlimit}')
    scope.write(f'{c}:PROBe {probe}')
    scope.write(f'{c}:IMPedance {impedance}')
    scope.write(f'{c}:UNIT {unit}')
    scope.write(f'{c}:INVert {"ON" if invert else "OFF"}')
    print(f"CH{n}: {scale_v:.3f} V/div, offset={offset_v:.3f} V, "
          f"{coupling}, probe={probe}x, {impedance}, bw={bwlimit}")

def get_channel(n):
    """
    Query all parameters for channel n.

    Returns
    -------
    dict with keys: display, scale_v, offset_v, coupling, bwlimit,
                    probe, impedance, unit, invert
    """
    c = f':CHANnel{n}'
    s = {
        'display':   _q(f'{c}:DISPlay?'),
        'scale_v':   _qf(f'{c}:SCALe?'),
        'offset_v':  _qf(f'{c}:OFFSet?'),
        'coupling':  _q(f'{c}:COUPling?'),
        'bwlimit':   _q(f'{c}:BWLimit?'),
        'probe':     _qf(f'{c}:PROBe?'),
        'impedance': _q(f'{c}:IMPedance?'),
        'unit':      _q(f'{c}:UNIT?'),
        'invert':    _q(f'{c}:INVert?'),
    }
    print(f"CH{n}: {s['scale_v']:.4f} V/div  offset={s['offset_v']:.4f} V  "
          f"{s['coupling']}  probe={s['probe']:.0f}x  "
          f"{s['impedance']}  bw={s['bwlimit']}  display={s['display']}")
    return s

def get_all_channels():
    """Query settings for all four channels. Returns dict keyed by channel number."""
    return {n: get_channel(n) for n in range(1, 5)}


# ══════════════════════════════════════════════════════════════════════════════
#  ACQUIRE
# ══════════════════════════════════════════════════════════════════════════════

def set_acquire(acq_type=ACQ_TYPE, averages=ACQ_AVERAGES, mdepth=ACQ_MDEPTH):
    """
    Configure the acquisition system.

    Parameters
    ----------
    acq_type : 'NORM', 'AVER', 'PEAK', 'HRES'
    averages : Number of averages (only used when acq_type='AVER')
    mdepth   : Memory depth — valid values: 1000, 10000, 100000, 1000000
    """
    scope.write(f':ACQuire:TYPE {acq_type}')
    if acq_type == 'AVER':
        scope.write(f':ACQuire:AVERages {int(averages)}')
    scope.write(f':ACQuire:MDEPth {int(mdepth)}')
    avg_str = f', averages={averages}' if acq_type == 'AVER' else ''
    print(f"Acquire: type={acq_type}{avg_str}, mdepth={mdepth}")

def get_acquire():
    """
    Query acquisition settings.

    Returns
    -------
    dict with keys: type, averages, mdepth, srate
    """
    srate = _qf(':ACQuire:SRATe?')
    s = {
        'type':     _q(':ACQuire:TYPE?'),
        'averages': _q(':ACQuire:AVERages?'),
        'mdepth':   _q(':ACQuire:MDEPth?'),
        'srate':    srate,
    }
    print(f"Acquire: type={s['type']}, averages={s['averages']}, "
          f"mdepth={s['mdepth']}, srate={srate/1e9:.1f} GS/s")
    return s


# ══════════════════════════════════════════════════════════════════════════════
#  TRIGGER
# ══════════════════════════════════════════════════════════════════════════════

def set_trigger_edge(
    source=TRIG_SRC,
    slope=TRIG_SLOPE,
    level_v=TRIG_LEVEL_V,
    sweep=TRIG_SWEEP,
    coupling=TRIG_COUPLING,
    holdoff_s=TRIG_HOLDOFF
):
    """
    Configure edge trigger.

    Parameters
    ----------
    source    : 'CHAN1'..'CHAN4', 'EXT', 'AC'
    slope     : 'POS', 'NEG', 'RFAL'
    level_v   : Trigger level in volts
    sweep     : 'AUTO', 'NORM', 'SING'
    coupling  : 'DC', 'AC', 'LFR', 'HFR'
    holdoff_s : Holdoff in seconds
    """
    scope.write(':TRIGger:MODE EDGE')
    scope.write(f':TRIGger:SWEep {sweep}')
    scope.write(f':TRIGger:COUPling {coupling}')
    scope.write(f':TRIGger:HOLDoff {holdoff_s:.6e}')
    scope.write(f':TRIGger:EDGe:SOURce {source}')
    scope.write(f':TRIGger:EDGe:SLOPe {slope}')
    scope.write(f':TRIGger:EDGe:LEVel {level_v:.6e}')
    print(f"Trigger: EDGE  src={source}  slope={slope}  "
          f"level={level_v:.3f} V  sweep={sweep}")

def set_trigger_pulse(
    source=TRIG_SRC,
    when='GRE',
    width_s=1e-6,
    level_v=TRIG_LEVEL_V,
    sweep=TRIG_SWEEP
):
    """
    Configure pulse width trigger.

    Parameters
    ----------
    source  : Trigger source
    when    : 'GRE' (>), 'LESS' (<), 'GLES' (inside), 'EQ', 'NEQ'
    width_s : Reference pulse width in seconds
    level_v : Trigger level in volts
    sweep   : 'AUTO', 'NORM', 'SING'
    """
    scope.write(':TRIGger:MODE PULSe')
    scope.write(f':TRIGger:SWEep {sweep}')
    scope.write(f':TRIGger:PULSe:SOURce {source}')
    scope.write(f':TRIGger:PULSe:WHEN {when}')
    scope.write(f':TRIGger:PULSe:WIDTh {width_s:.6e}')
    scope.write(f':TRIGger:PULSe:LEVel {level_v:.6e}')
    print(f"Trigger: PULSE  src={source}  when={when}  "
          f"width={width_s*1e6:.3f} µs  level={level_v:.3f} V")

def get_trigger():
    """
    Query current trigger settings.

    Returns
    -------
    dict with keys: mode, sweep, status, coupling, holdoff, src, slope, level
    """
    s = {
        'mode':     _q(':TRIGger:MODE?'),
        'sweep':    _q(':TRIGger:SWEep?'),
        'status':   _q(':TRIGger:STATus?'),
        'coupling': _q(':TRIGger:COUPling?'),
        'holdoff':  _q(':TRIGger:HOLDoff?'),
        'src':      _q(':TRIGger:EDGe:SOURce?'),
        'slope':    _q(':TRIGger:EDGe:SLOPe?'),
        'level':    _q(':TRIGger:EDGe:LEVel?'),
    }
    print(f"Trigger: {s['mode']}  src={s['src']}  slope={s['slope']}  "
          f"level={s['level']} V  sweep={s['sweep']}  status={s['status']}")
    return s


# ══════════════════════════════════════════════════════════════════════════════
#  MEASUREMENTS
#  Returns np.nan when the scope returns its 9.9E+37 invalid sentinel.
# ══════════════════════════════════════════════════════════════════════════════

def measure(item, channel=1):
    """
    Query a single measurement on the specified channel.

    Parameters
    ----------
    item    : Key from MEAS_ITEMS (e.g. 'frequency') or a raw mnemonic
              (e.g. 'FREQuency'). See MEAS_ITEMS in the config block.
    channel : Channel number 1–4

    Returns
    -------
    value : float — np.nan if the measurement is unavailable
    """
    mnem = MEAS_ITEMS.get(item, item)
    raw  = _q(f':MEASure:ITEM? {mnem},CHANnel{channel}')
    if raw is None:
        return np.nan
    val = float(raw)
    return np.nan if val >= MEAS_INVALID * 0.9 else val

def measure_all(channel=1):
    """
    Query every measurement in MEAS_ITEMS for the given channel.
    Prints a table and returns a dict.

    Parameters
    ----------
    channel : Channel number 1–4

    Returns
    -------
    dict : {item_name: float_or_nan}
    """
    results = {}
    print(f"\nMeasurements — CH{channel}:")
    for name in MEAS_ITEMS:
        val = measure(name, channel)
        results[name] = val
        if not np.isnan(val):
            print(f"  {name:<14} = {val:.6g}")
        else:
            print(f"  {name:<14} = (no signal)")
    return results


# ══════════════════════════════════════════════════════════════════════════════
#  WAVEFORM CAPTURE
#
#  Uses BYTE format. Preamble field order:
#    format, type, points, count, XI, XO, XR, YI, YO, YR
#  Voltage conversion:  V = (raw_byte - YREFerence) * YINCrement + YORigin
#  For BYTE format:     YREFerence = 128 (confirmed)
# ══════════════════════════════════════════════════════════════════════════════

def capture_waveform(channel=1, points=WAV_POINTS):
    """
    Capture waveform data from the specified channel using BYTE format.

    Parameters
    ----------
    channel    : Channel number 1–4
    points     : Number of points (max = ACQ_MDEPTH)
    stop_first : Stop the scope before reading if True

    Returns
    -------
    t_s  : np.ndarray — time axis in seconds
    v_V  : np.ndarray — voltage in volts
    info : dict — preamble values for reference
    """
    # Scope must be stopped before points and format can be changed
    scope.write(':STOP')
    time.sleep(0.3)

    scope.write(f':WAVeform:SOURce CHANnel{channel}')
    # MAXimum mode returns all acquired points up to MDEPth.
    # NORMal mode only returns screen display points (~1000 cap).
    scope.write(':WAVeform:MODE MAXimum')
    time.sleep(0.05)
    # Verify MAXimum was accepted — fall back to NORMal if not
    mode_rb = scope.query(':WAVeform:MODE?').strip()
    if mode_rb not in ('MAX', 'MAXIMUM', 'MAXimum'):
        print("  ⚠️  MAXimum mode not accepted — falling back to NORMal (1000 pt cap).")
        scope.write(':WAVeform:MODE NORMal')
    scope.write(':WAVeform:FORMat BYTE')
    scope.write(f':WAVeform:POINts {int(points)}')
    time.sleep(0.1)

    # Verify the scope accepted the points value
    actual_pts = scope.query(':WAVeform:POINts?').strip()
    if int(actual_pts) != int(points):
        print(f"  ⚠️  Requested {points} pts but scope set {actual_pts} pts. "
              f"Check that points ≤ ACQ_MDEPTH ({ACQ_MDEPTH}).")

    # Read all scaling from a single preamble query
    preamble = _q(':WAVeform:PREamble?')
    if preamble:
        p    = [float(x) for x in preamble.split(',')]
        xi, xo, yi, yo, yref = p[4], p[5], p[7], p[8], p[9]
    else:
        # Fallback: individual queries
        xi   = _qf(':WAVeform:XINCrement?')
        xo   = _qf(':WAVeform:XORigin?')
        yi   = _qf(':WAVeform:YINCrement?')
        yo   = _qf(':WAVeform:YORigin?')
        yref = _qf(':WAVeform:YREFerence?')

    # Read raw BYTE data
    scope.write(':WAVeform:DATA?')
    raw      = scope.read_raw()
    n_digits = int(chr(raw[1]))
    n_bytes  = int(raw[2:2+n_digits])
    data     = np.frombuffer(raw[2+n_digits:2+n_digits+n_bytes], dtype=np.uint8)

    v_V = (data.astype(float) - yref) * yi + yo
    t_s = xo + xi * np.arange(len(v_V))

    info = {'xi': xi, 'xo': xo, 'yi': yi, 'yo': yo,
            'yref': yref, 'points': len(v_V), 'channel': channel}
    print(f"CH{channel}: {len(v_V)} pts  "
          f"Vmin={v_V.min():.4f} V  Vmax={v_V.max():.4f} V  "
          f"Vpp={v_V.max()-v_V.min():.4f} V")
    return t_s, v_V, info


def save_waveform(channel=1, points=WAV_POINTS, filename=None):
    """
    Capture waveform and save to CSV.

    Returns
    -------
    df : DataFrame with columns [time_s, voltage_V]
    """
    t_s, v_V, _ = capture_waveform(channel, points)
    df = pd.DataFrame({'time_s': t_s, 'voltage_V': v_V})
    if filename is None:
        ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = os.path.join(OUTPUT_DIR, f'ch{channel}_waveform_{ts}.csv')
    df.to_csv(filename, index=False)
    print(f"Saved → {filename}")
    scope.write(':RUN')
    return df


def plot_waveform(channel=1, points=WAV_POINTS, title=None, save_png=False):
    """
    Capture and plot a waveform inline.

    Returns
    -------
    (t_s, v_V) numpy arrays
    """
    t_s, v_V, _ = capture_waveform(channel, points)
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(t_s * 1e6, v_V, linewidth=0.7)
    ax.set_xlabel('Time (µs)')
    ax.set_ylabel('Voltage (V)')
    ax.set_title(title or f'MHO5104 — CH{channel}')
    ax.grid(True, alpha=0.4)
    plt.tight_layout()
    if save_png:
        ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        fname = os.path.join(OUTPUT_DIR, f'ch{channel}_plot_{ts}.png')
        plt.savefig(fname, dpi=150)
        print(f"Plot saved → {fname}")
    plt.show()
    scope.write(':RUN')
    return t_s, v_V


def plot_multi_channel(channels=(1, 2), points=WAV_POINTS, title=None):
    """
    Capture and overlay multiple channels on one axes.

    Parameters
    ----------
    channels : tuple of channel numbers, e.g. (1, 2) or (1, 2, 3, 4)
    """
    fig, ax = plt.subplots(figsize=(11, 5))
    for ch_n in channels:
        try:
            t_s, v_V, _ = capture_waveform(ch_n, points)
            ax.plot(t_s * 1e6, v_V, linewidth=0.7, label=f'CH{ch_n}')
        except Exception as e:
            print(f"  CH{ch_n} skipped: {e}")
    ax.set_xlabel('Time (µs)')
    ax.set_ylabel('Voltage (V)')
    ax.set_title(title or 'MHO5104 — Multi-channel')
    ax.legend()
    ax.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.show()
    scope.write(':RUN')


# ══════════════════════════════════════════════════════════════════════════════
#  SCREENSHOT
#
#  The scope sends a 3,072,054-byte 1280×800 24-bit BMP with an IEEE 488.2
#  block header. read_raw() stops early at the USB-TMC EOM bit regardless of
#  the declared byte count, so we use query_binary_values() which loops on the
#  declared size. The raw BMP is then converted to PNG via Pillow.
# ══════════════════════════════════════════════════════════════════════════════

def screenshot(filename=None):
    """
    Capture a PNG screenshot of the scope display.

    The scope outputs a raw BMP which is converted to PNG automatically.
    Requires Pillow: pip install Pillow

    Parameters
    ----------
    filename : str or None — auto-generated with timestamp if None

    Returns
    -------
    filename : str — path of the saved PNG
    """
    if filename is None:
        ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = os.path.join(OUTPUT_DIR, f'screenshot_{ts}.png')
    # Ensure .png extension regardless of what was passed
    filename = os.path.splitext(filename)[0] + '.png'

    old_timeout = scope.timeout
    scope.timeout = 60000   # 60 s — 3 MB BMP at USB 2.0 speeds

    raw_bmp = None
    try:
        # query_binary_values loops on the declared IEEE block byte count
        # rather than stopping at the USB-TMC EOM bit
        img_bytes = scope.query_binary_values(
            ':DISPlay:DATA?',
            datatype      = 'B',          # unsigned byte
            is_big_endian = False,
            container     = bytearray,
            chunk_size    = 1024 * 1024   # 1 MB per internal chunk
        )
        raw_bmp = bytes(img_bytes)
        print(f"  Received {len(raw_bmp):,} bytes via query_binary_values")
    except Exception as e:
        print(f"  query_binary_values failed ({e}), trying loop fallback...")
        raw_bmp = _screenshot_loop_read()
    finally:
        scope.timeout = old_timeout

    if raw_bmp is None or len(raw_bmp) < 54:   # 54 = minimum BMP header size
        print("Screenshot failed — no data received.")
        return None

    # Convert BMP bytes → PNG via Pillow
    img = Image.open(io.BytesIO(raw_bmp))
    img.save(filename, format='PNG', optimize=True)
    print(f"Screenshot → {filename}  "
          f"({img.size[0]}×{img.size[1]} px, {os.path.getsize(filename):,} bytes)")
    return filename


def _screenshot_loop_read():
    """
    Fallback screenshot read: parse the declared byte count from the IEEE
    block header in the first chunk, then keep calling read_raw() until
    all declared bytes are accumulated.

    Returns
    -------
    raw_bmp : bytes or None
    """
    scope.timeout = 60000
    scope.write(':DISPlay:DATA?')
    time.sleep(1.0)

    try:
        first = scope.read_raw()
    except Exception as e:
        print(f"  First read failed: {e}")
        return None

    if not first or chr(first[0]) != '#':
        print(f"  Unexpected response start: {list(first[:8])}")
        return None

    n_digits   = int(chr(first[1]))
    n_declared = int(first[2:2+n_digits])
    header_len = 2 + n_digits
    all_data   = bytearray(first[header_len:])
    print(f"  Header: {n_declared:,} bytes declared, "
          f"first chunk {len(all_data):,} bytes")

    while len(all_data) < n_declared:
        remaining = n_declared - len(all_data)
        print(f"  Reading {remaining:,} remaining bytes...")
        try:
            more = scope.read_raw()
            if not more:
                break
            all_data.extend(more)
        except pyvisa.errors.VisaIOError:
            print(f"  Read stopped at {len(all_data):,} of {n_declared:,} bytes")
            break

    scope.timeout = TIMEOUT_MS
    return bytes(all_data[:n_declared])


# ══════════════════════════════════════════════════════════════════════════════
#  MATH  (must use :MATH1: prefix — bare :MATH: is not valid on this firmware)
# ══════════════════════════════════════════════════════════════════════════════

def set_math(
    math_n=1,
    operator='ADD',
    source1='CHAN1',
    source2='CHAN2',
    scale=1.0,
    offset=0.0,
    display=True,
    fft_center_hz=None,
    fft_window='HANN',
    fft_unit='DB',
):
    """
    Configure a MATH channel (1–4).

    Parameters
    ----------
    math_n      : Math channel number 1–4
    operator    : 'ADD', 'SUBT', 'MULT', 'DIV', 'FFT',
                  'AND', 'OR', 'XOR', 'NOT',
                  'INTG', 'DIFF', 'SQRT', 'LOG', 'LN', 'EXP', 'ABS'
    source1     : Primary source — 'CHAN1'..'CHAN4'
    source2     : Secondary source — 'CHAN1'..'CHAN4' (non-FFT operators)
    scale       : Vertical scale (non-FFT operators)
    offset      : Vertical offset (non-FFT operators)
    display     : True = show the math channel on screen
    fft_center_hz : FFT center frequency in Hz (None = leave unchanged).
                    Controlled via :MATH{n}:FFT:HCEN — confirmed working.
    # Remove fft_span_hz entirely — use set_math_fft_span() instead.
    fft_window  : FFT window — 'RECT', 'HANN', 'BLAC', 'FLAT'
    fft_unit    : FFT vertical unit — 'DB', 'VRMS'
    """
    m = f':MATH{math_n}'
    scope.write(f'{m}:DISPlay {"ON" if display else "OFF"}')
    scope.write(f'{m}:OPERator {operator}')
    scope.write(f'{m}:SOURce1 {source1}')

    if operator == 'FFT':
        scope.write(f'{m}:FFT:SOURce {source1}')
        scope.write(f'{m}:FFT:WINDow {fft_window}')
        scope.write(f'{m}:FFT:UNIT {fft_unit}')
        if fft_center_hz is not None:
            scope.write(f'{m}:FFT:HCEN {fft_center_hz:.6e}')
        time.sleep(0.1)
        # Readback confirmed values
        rb_center = _qf(f'{m}:FFT:HCEN?')
        rb_window = _q(f'{m}:FFT:WINDow?')
        rb_unit   = _q(f'{m}:FFT:UNIT?')
        rb_src    = _q(f'{m}:FFT:SOURce?')
        c_str = f"{rb_center/1e6:.4g} MHz" if rb_center is not None else "n/a"
        print(f"MATH{math_n}: FFT  src={rb_src}  center={c_str}  "
              f"window={rb_window}  unit={rb_unit}")
        print(f"  Note: FFT span is controlled by timebase scale.")
    else:
        scope.write(f'{m}:SOURce2 {source2}')
        scope.write(f'{m}:SCALe {scale:.6e}')
        scope.write(f'{m}:OFFSet {offset:.6e}')
        print(f"MATH{math_n}: {operator}({source1}, {source2})  "
              f"scale={scale}  offset={offset}  display={display}")

    check_error()


def set_math_fft_center(math_n=1, center_hz=None):
    """
    Update only the FFT center frequency (horizontal pan) on an FFT math channel.
    FFT span is controlled by the timebase — use set_timebase() to change it.

    Parameters
    ----------
    math_n    : Math channel number 1–4
    center_hz : Center frequency in Hz
    """
    if center_hz is None:
        print("No center frequency specified.")
        return
    m = f':MATH{math_n}'
    scope.write(f'{m}:FFT:HCEN {center_hz:.6e}')
    time.sleep(0.1)
    rb = _qf(f'{m}:FFT:HCEN?')
    c_str = f"{rb/1e6:.4g} MHz" if rb is not None else "unavailable"
    print(f"MATH{math_n} FFT center: {c_str}")

def set_math_fft_span(math_n=1, span_hz=None, mdepth=None):
    """
    Set the FFT frequency span by adjusting the timebase scale.
    FFT span has no dedicated SCPI command on this firmware — it is
    determined entirely by: span = MDEPth / (timebase_s × 20)

    Parameters
    ----------
    math_n   : Math channel number (informational only — span affects all)
    span_hz  : Desired FFT span in Hz
    mdepth   : Memory depth to use (None = query current value)

    Returns
    -------
    dict with keys: timebase_s, actual_span_hz, freq_resolution_hz
    """
    import math as _math

    SCREEN_DIVISIONS = 10

    if span_hz is None:
        print("No span specified.")
        return

    # Query current memory depth if not provided
    if mdepth is None:
        rb = _q(':ACQuire:MDEPth?')
        mdepth = int(float(rb)) if rb else ACQ_MDEPTH

    # Required timebase: timebase_s = mdepth / (span_hz × 20)
    required_tb = mdepth / (span_hz * SCREEN_DIVISIONS * 2)

    # Snap to nearest standard 1-2-5 timebase step
    def _snap_125(val):
        exp = _math.floor(_math.log10(val))
        best = None
        for m in [1, 2, 5]:
            for e in [exp - 1, exp, exp + 1]:
                candidate = m * 10 ** e
                if best is None or abs(candidate - val) < abs(best - val):
                    best = candidate
        return best

    timebase_s = _snap_125(required_tb)

    # Actual span achieved with snapped timebase
    actual_span_hz    = mdepth / (timebase_s * SCREEN_DIVISIONS * 2)
    freq_res_hz       = actual_span_hz / (mdepth / 2)

    print(f"MATH{math_n} FFT span:")
    print(f"  Requested span     : {span_hz/1e6:.4g} MHz")
    print(f"  Required timebase  : {required_tb*1e6:.4g} µs/div")
    print(f"  Snapped timebase   : {timebase_s*1e6:.4g} µs/div")
    print(f"  Actual span        : {actual_span_hz/1e6:.4g} MHz")
    print(f"  Freq resolution    : {freq_res_hz:.4g} Hz/pt")
    print(f"  Memory depth       : {mdepth:,} pts")

    # Apply timebase
    scope.write(f':TIMebase:MAIN:SCALe {timebase_s:.10e}')

    return {
        'timebase_s':       timebase_s,
        'actual_span_hz':   actual_span_hz,
        'freq_resolution_hz': freq_res_hz,
    }

def math_on(math_n):
    """Enable display of MATH channel math_n (1–4)."""
    scope.write(f':MATH{math_n}:DISPlay ON')
    print(f"MATH{math_n}: ON")


def math_off(math_n):
    """Disable display of MATH channel math_n (1–4)."""
    scope.write(f':MATH{math_n}:DISPlay OFF')
    print(f"MATH{math_n}: OFF")


def get_math(math_n=None):
    """
    Query MATH channel settings.

    Parameters
    ----------
    math_n : Channel number 1–4, or None to query all four

    Returns
    -------
    dict (single channel) or dict of dicts keyed by channel number
    """
    def _query_one(n):
        m  = f':MATH{n}'
        op = _q(f'{m}:OPERator?')
        s  = {
            'display':  _q(f'{m}:DISPlay?'),
            'operator': op,
            'source1':  _q(f'{m}:SOURce1?'),
            'source2':  _q(f'{m}:SOURce2?'),
            'scale':    _q(f'{m}:SCALe?'),
            'offset':   _q(f'{m}:OFFSet?'),
        }
        status = 'ON ' if s['display'] in ('1', 'ON') else 'OFF'
        if op == 'FFT':
            rb_center = _qf(f'{m}:FFT:HCEN?')
            s['fft_center'] = rb_center
            s['fft_window'] = _q(f'{m}:FFT:WINDow?')
            s['fft_unit']   = _q(f'{m}:FFT:UNIT?')
            c_str = f"{rb_center/1e6:.4g} MHz" if rb_center is not None else "n/a"
            print(f"  MATH{n} [{status}]  FFT  src={s['source1']}  "
                  f"center={c_str}  window={s['fft_window']}  unit={s['fft_unit']}")
        else:
            print(f"  MATH{n} [{status}]  {op}({s['source1']}, {s['source2']})  "
                  f"scale={s['scale']}  offset={s['offset']}")
        return s

    if math_n is not None:
        return _query_one(math_n)

    print("── MATH channels ─────────────────────────────────────")
    return {n: _query_one(n) for n in range(1, 5)}


# ══════════════════════════════════════════════════════════════════════════════
#  CURSOR
#  Individual AX/BX/AY/BY position queries are unreliable on this firmware.
#  :CURSor:ITEM? returns a JSON list with the current cursor state.
# ══════════════════════════════════════════════════════════════════════════════

def cursor_on(mode='MAN'):
    """
    Enable cursors.

    Parameters
    ----------
    mode : 'MAN', 'TRACk', 'AUTO', 'XY'
    """
    scope.write(f':CURSor:MODE {mode}')
    print(f"Cursor: {mode}")

def cursor_off():
    """Disable cursors."""
    scope.write(':CURSor:MODE OFF')
    print("Cursor: OFF")

def cursor_get_item():
    """
    Query the raw cursor state as a JSON string.
    Parse with json.loads() if programmatic access is needed.

    Returns
    -------
    raw JSON string
    """
    raw = _q(':CURSor:ITEM?')
    print(f"Cursor ITEM: {raw}")
    return raw


# ══════════════════════════════════════════════════════════════════════════════
#  AWG — Built-in Function Generator (:SOURce1: subsystem)
# ══════════════════════════════════════════════════════════════════════════════

def set_awg_function(func=AWG_FUNCTION):
    """
    Set AWG waveform type.

    Parameters
    ----------
    func : 'SIN', 'SQU', 'RAMP', 'PULS', 'NOIS', 'DC', 'ARB'
    """
    scope.write(f':SOURce1:FUNCtion {func}')
    print(f"AWG function: {func}")

def set_awg_frequency(freq_hz=AWG_FREQ_HZ):
    """
    Set AWG output frequency in Hz.

    Parameters
    ----------
    freq_hz : Frequency in Hz
    """
    scope.write(f':SOURce1:FREQuency {freq_hz:.6e}')
    print(f"AWG frequency: {freq_hz:.3f} Hz")

def get_awg():
    """
    Query AWG settings.

    Returns
    -------
    dict with keys: function, frequency
    """
    s = {
        'function':  _q(':SOURce1:FUNCtion?'),
        'frequency': _qf(':SOURce1:FREQuency?'),
    }
    print(f"AWG: func={s['function']}  freq={s['frequency']:.3f} Hz")
    return s

def scope_print_settings():
    """
    Query and display all current scope settings without changing anything.
    Covers timebase, all four channels, acquire, trigger, math, AWG,
    and trigger/acquire status.
    """
    print("=" * 60)
    print("  MHO5104 — Current Settings")
    print("=" * 60)

    # ── Timebase ──────────────────────────────────────────────
    print("\n── Timebase ──────────────────────────────────────────────")
    scale  = _qf(':TIMebase:MAIN:SCALe?')
    offset = _qf(':TIMebase:MAIN:OFFSet?')
    mode   = _q(':TIMebase:MODE?')
    print(f"  Mode     : {mode}")
    print(f"  Scale    : {scale*1e6:.4g} µs/div")
    print(f"  Offset   : {offset*1e6:.4g} µs")
    if mode != 'MAIN':
        dscale  = _qf(':TIMebase:DELay:SCALe?')
        doffset = _qf(':TIMebase:DELay:OFFSet?')
        denab   = _q(':TIMebase:DELay:ENABle?')
        print(f"  Delay    : enabled={denab}  scale={dscale*1e6:.4g} µs  "
              f"offset={doffset*1e6:.4g} µs")

    # ── Channels ──────────────────────────────────────────────
    print("\n── Channels ──────────────────────────────────────────────")
    for n in range(1, 5):
        c      = f':CHANnel{n}'
        disp   = _q(f'{c}:DISPlay?')
        scale  = _qf(f'{c}:SCALe?')
        offset = _qf(f'{c}:OFFSet?')
        coup   = _q(f'{c}:COUPling?')
        bw     = _q(f'{c}:BWLimit?')
        probe  = _qf(f'{c}:PROBe?')
        imp    = _q(f'{c}:IMPedance?')
        unit   = _q(f'{c}:UNIT?')
        inv    = _q(f'{c}:INVert?')
        status = 'ON ' if disp in ('1', 'ON') else 'OFF'
        print(f"  CH{n} [{status}]  {scale:.4g} V/div  "
              f"offset={offset:.4g} V  {coup}  "
              f"probe={probe:.0f}x  {imp}  bw={bw}  "
              f"invert={inv}  unit={unit}")

    # ── Acquire ───────────────────────────────────────────────
    print("\n── Acquire ───────────────────────────────────────────────")
    acq_type = _q(':ACQuire:TYPE?')
    averages = _q(':ACQuire:AVERages?')
    mdepth   = _q(':ACQuire:MDEPth?')
    srate    = _qf(':ACQuire:SRATe?')
    print(f"  Type     : {acq_type}"
          + (f"  (averages={averages})" if acq_type == 'AVER' else ""))
    print(f"  Mem depth: {int(float(mdepth)):,} pts")
    print(f"  Sample rate: {srate/1e9:.2f} GS/s")

    # ── Trigger ───────────────────────────────────────────────
    print("\n── Trigger ───────────────────────────────────────────────")
    tmode   = _q(':TRIGger:MODE?')
    sweep   = _q(':TRIGger:SWEep?')
    status  = _q(':TRIGger:STATus?')
    coup    = _q(':TRIGger:COUPling?')
    holdoff = _qf(':TRIGger:HOLDoff?')
    print(f"  Mode     : {tmode}  sweep={sweep}  status={status}")
    print(f"  Coupling : {coup}  holdoff={holdoff*1e9:.1f} ns")
    if tmode == 'EDGE':
        src   = _q(':TRIGger:EDGe:SOURce?')
        slope = _q(':TRIGger:EDGe:SLOPe?')
        level = _qf(':TRIGger:EDGe:LEVel?')
        print(f"  Edge     : src={src}  slope={slope}  level={level:.4g} V")
    elif tmode == 'PULS':
        src   = _q(':TRIGger:PULSe:SOURce?')
        when  = _q(':TRIGger:PULSe:WHEN?')
        width = _qf(':TRIGger:PULSe:WIDTh?')
        level = _qf(':TRIGger:PULSe:LEVel?')
        print(f"  Pulse    : src={src}  when={when}  "
              f"width={width*1e6:.4g} µs  level={level:.4g} V")

    # ── Math ──────────────────────────────────────────────────
    print("\n── Math1 ──────────────────────────────────────────────────")
    mdisp = _q(':MATH1:DISPlay?')
    mop   = _q(':MATH1:OPERator?')
    ms1   = _q(':MATH1:SOURce1?')
    ms2   = _q(':MATH1:SOURce2?')
    mscl  = _q(':MATH1:SCALe?')
    moff  = _q(':MATH1:OFFSet?')
    status = 'ON ' if mdisp in ('1', 'ON') else 'OFF'
    print(f"  MATH1 [{status}]  {mop}({ms1}, {ms2})  "
          f"scale={mscl}  offset={moff}")
    print("\n── Math2 ──────────────────────────────────────────────────")
    mdisp = _q(':MATH2:DISPlay?')
    mop   = _q(':MATH2:OPERator?')
    ms1   = _q(':MATH2:SOURce1?')
    ms2   = _q(':MATH2:SOURce2?')
    mscl  = _q(':MATH2:SCALe?')
    moff  = _q(':MATH2:OFFSet?')
    status = 'ON ' if mdisp in ('1', 'ON') else 'OFF'
    print(f"  MATH2 [{status}]  {mop}({ms1}, {ms2})  "
          f"scale={mscl}  offset={moff}")

    # ── AWG ───────────────────────────────────────────────────
    print("\n── AWG ───────────────────────────────────────────────────")
    afunc = _q(':SOURce1:FUNCtion?')
    afreq = _qf(':SOURce1:FREQuency?')
    print(f"  Function : {afunc}")
    print(f"  Frequency: {afreq:.6g} Hz")

    print("\n" + "=" * 60)

def scope_setup_for_nyquist(
    freq_hz,
    channel=1,
    points=100000,
    coupling=CH_COUPLING,
    probe=CH_PROBE,
    trigger_level_v=0.0,
    trigger_slope=TRIG_SLOPE,
    verbose=True
):
    """
    Configure the scope for a known input frequency using 100k points.

    The sample interval is set to the first standard 1-2-5 step that falls
    strictly below the Nyquist interval (1 / 2*freq_hz). The total time window
    and timebase are then derived directly from that interval × point count.

    Parameters
    ----------
    freq_hz         : Known input frequency in Hz
    channel         : Channel to configure and trigger on (1–4)
    points          : Memory depth — snapped to nearest valid value
                      (1000 / 10000 / 100000 / 1000000, default 100000)
    coupling        : 'DC', 'AC', 'GND'
    probe           : Probe attenuation ratio
    trigger_level_v : Trigger level in volts
    trigger_slope   : 'POS', 'NEG', 'RFAL'
    verbose         : Print calculation summary if True

    Returns
    -------
    dict with keys:
        freq_hz, dt_s, sample_rate, nyquist_margin,
        total_time_s, timebase_s, mdepth, n_cycles
    """
    import math

    SCREEN_DIVISIONS = 10
    SCOPE_MAX_RATE   = 4e9   # 4 GS/s hardware limit
    VALID_MDEPTHS    = [1000, 10000, 100000, 1000000]

    # Snap requested points to nearest valid memory depth
    mdepth = min(VALID_MDEPTHS, key=lambda x: abs(x - points))

    # ── Step 1: Nyquist interval ───────────────────────────────────────────
    # Maximum allowable sample spacing to avoid aliasing
    nyquist_interval_s = 1.0 / (2.0 * freq_hz)

    # ── Step 2: First standard 1-2-5 step strictly below Nyquist interval ─
    def _first_step_below(val):
        """Largest standard 1-2-5 step that is strictly less than val."""
        exp = math.floor(math.log10(val))
        for m in [5, 2, 1]:
            candidate = m * 10 ** exp
            if candidate < val:
                return candidate
        return 5 * 10 ** (exp - 1)   # drop one decade if needed

    dt_s        = _first_step_below(nyquist_interval_s)
    sample_rate = 1.0 / dt_s

    # ── Step 3: Warn if sample rate exceeds scope hardware limit ──────────
    hardware_limited = False
    if sample_rate > SCOPE_MAX_RATE:
        print(f"  ⚠️  Calculated sample rate {sample_rate/1e9:.2f} GS/s exceeds "
              f"scope maximum {SCOPE_MAX_RATE/1e9:.0f} GS/s.")
        print(f"      Hardware will cap at {SCOPE_MAX_RATE/1e9:.0f} GS/s — "
              f"signal may alias.")
        sample_rate = SCOPE_MAX_RATE
        dt_s        = 1.0 / sample_rate
        hardware_limited = True

    # ── Step 4: Derive time window and timebase ────────────────────────────
    total_time_s = mdepth * dt_s
    timebase_s   = total_time_s / SCREEN_DIVISIONS
    n_cycles     = total_time_s * freq_hz
    margin       = sample_rate / (2.0 * freq_hz)

    # ── Step 5: Print summary ──────────────────────────────────────────────
    if verbose:
        print("=" * 55)
        print("  Frequency Setup")
        print("=" * 55)
        print(f"  Input frequency    : {freq_hz:.6g} Hz  "
              f"({freq_hz/1e6:.4g} MHz)")
        print(f"  Period             : {1/freq_hz*1e9:.4g} ns")
        print()
        print(f"  Nyquist interval   : {nyquist_interval_s*1e9:.4g} ns")
        print(f"  Sample interval    : {dt_s*1e9:.4g} ns"
              + ("  (hardware limited)" if hardware_limited else ""))
        print(f"  Sample rate        : {sample_rate/1e6:.4g} MS/s  "
              f"({margin:.2f}× Nyquist)")
        print()
        print(f"  Memory depth       : {mdepth:,} pts")
        print(f"  Total window       : {total_time_s*1e3:.4g} ms")
        print(f"  Timebase scale     : {timebase_s*1e6:.4g} µs/div")
        print(f"  Cycles on screen   : {n_cycles:.1f}")
        print("=" * 55)

    # ── Step 6: Apply to scope ─────────────────────────────────────────────
    scope.write(':STOP')
    time.sleep(0.1)

    scope.write(':TIMebase:MODE MAIN')
    scope.write(f':TIMebase:MAIN:SCALe {timebase_s:.10e}')
    scope.write(':TIMebase:MAIN:OFFSet 0')

    scope.write(f':CHANnel{channel}:DISPlay ON')
    scope.write(f':CHANnel{channel}:COUPling {coupling}')
    scope.write(f':CHANnel{channel}:PROBe {probe}')

    scope.write(':ACQuire:TYPE NORM')
    scope.write(f':ACQuire:MDEPth {mdepth}')

    scope.write(':TRIGger:MODE EDGE')
    scope.write(':TRIGger:SWEep AUTO')
    scope.write(f':TRIGger:EDGe:SOURce CHAN{channel}')
    scope.write(f':TRIGger:EDGe:SLOPe {trigger_slope}')
    scope.write(f':TRIGger:EDGe:LEVel {trigger_level_v:.6e}')

    scope.write(':RUN')
    time.sleep(0.2)

    if verbose:
        print(f"\n  Settings applied — CH{channel} running.")

    return {
        'freq_hz':       freq_hz,
        'dt_s':          dt_s,
        'sample_rate':   sample_rate,
        'nyquist_margin':margin,
        'total_time_s':  total_time_s,
        'timebase_s':    timebase_s,
        'mdepth':        mdepth,
        'n_cycles':      n_cycles,
    }
print("MHO5104 functions defined — ready.")

## 🧪 Examples

In [ ]:
# ── Nyquist sampling ─────────────────────────────────────────────────
known_frequency = 90e6
cfg = scope_setup_for_nyquist(known_frequency, channel=1, probe=1)


In [ ]:
# ── Example 1: Configure and run ──────────────────────────────────────────────
set_timebase(scale_s=100e-9)               # 50 ns/div
set_channel(1, scale_v=1.0, probe=1)
set_acquire(acq_type='NORM', mdepth=10000)
set_trigger_edge(source='CHAN1', slope='POS', level_v=0.0)
scope_run()

In [ ]:
# ── Example 2: All measurements on CH1 ───────────────────────────────────────
results = measure_all(channel=1)
print(f"\nFrequency : {results['frequency']:.6g} Hz")
print(f"Vpp       : {results['vamp']:.4f} V")
print(f"Rise time : {results['rise_time']*1e9:.2f} ns")

In [ ]:
# ── Example 3: Capture and plot CH1 ──────────────────────────────────────────
t, v = plot_waveform(channel=1, points=10000,
                     title='MHO5104 — CH1', save_png=True)

In [ ]:
# ── Example 4: Save waveform to CSV ──────────────────────────────────────────
df = save_waveform(channel=1, points=10000)
display(df.head())

In [ ]:
# ── Example 5: Multi-channel overlay ─────────────────────────────────────────
channel_on(2)
plot_multi_channel(channels=(1, 2), points=1000)

In [ ]:
# ── Example 6: Screenshot → PNG ──────────────────────────────────────────────
fname = screenshot()
if fname:
    print(f"Open {fname} to view the scope display")

In [ ]:
# ── Example: Math channels ────────────────────────────────────────────────────

# Step 1: Set memory depth first — span calculation depends on it
set_acquire(mdepth=100000)

# Step 2: Configure FFT on MATH1
set_math(
    math_n        = 1,
    operator      = 'FFT',
    source1       = 'CHAN1',
    fft_center_hz = 90e6,
    fft_window    = 'HANN',
    fft_unit      = 'DB',
)

# Step 3: Set FFT span via timebase (e.g. 0–50 MHz)
#set_math_fft_span(math_n=1, span_hz=1e6)

# Step 4: Pan the center without changing span
#set_math_fft_center(math_n=1, center_hz=25e6)

# Typical use — 10 MHz signal, show 0–20 MHz with 100k pts
# set_math_fft_span(math_n=1, span_hz=20e6)
# set_math_fft_center(math_n=1, center_hz=10e6)

In [ ]:
# ── Example 8: AWG output ─────────────────────────────────────────────────────
set_awg_function('SIN')
set_awg_frequency(10e3)    # 10 kHz
get_awg()

In [ ]:
# ── Example 9: Single-shot capture ───────────────────────────────────────────
scope_single()
status = scope_wait_trigger(timeout_s=10)
if status:
    t, v = plot_waveform(channel=1, points=10000,
                         title='Single shot', stop_first=False)

In [ ]:
# ── Example 10: Status and error check ───────────────────────────────────────
scope_status()
check_error()

In [ ]:
# ── Example 11: Read Settings ─────────────────────────────────────────────────
scope_print_settings()

---
## 🔌 Cleanup

In [ ]:
scope.write(':RUN')
scope.close()
rm.close()
print("MHO5104 connection closed.")